# Memory Management Software Techniques

## 🟢 Overlay Structure
An overlay structure is a way to run a program that is too large to fit into memory all at once.

---
### Core idea
You split the program into parts:
- a resident/root part that is always kept in memory
- several overlay parts that are loaded only when needed

The overlay parts share the **same memory area**, so one overlay replaces another.

So instead of keeping the whole program in RAM, you keep:
- the essential part always present
- only the currently needed module loaded
---
### Simple picture
Imagine memory like this:
- OS
- root / main program
- overlay area

At one moment, the overlay area may contain:
- overlay A
Later, it may be replaced by:
- overlay B

Both cannot be in that area at the same time, but that is okay if the program never needs them simultaneously.

---
### Why it was used
Overlays were used when:
- RAM was very small
- programs were getting larger
- hardware virtual memory was unavailable or limited

It let programmers run bigger applications than physical memory would normally allow.

---
### Example intuition
Suppose a program has:
- main menu
- report generator
- graphics module
- file converter

If the user never runs report generation and graphics at the same exact time, those modules can be stored as different overlays.

Memory might look like:
- permanent main/control code
- overlay region containing report generator

Then later:
- permanent main/control code
- overlay region containing graphics module

The graphics module overwrites the report generator in memory.

---
### Important property
Overlay parts are usually mutually exclusive:
- if module A is loaded, module B is not needed at the same time
- so both can share the same memory space

That is the heart of overlay design.

---
### Terms to know
**Root / resident part** - The portion of the program that always stays in memory.
It usually:
- controls execution
- decides when to load overlays
- contains common routines

**Overlay area**
- The memory region where temporary modules are loaded.

**Overlay manager / loader mechanism**
- loads the correct overlay
- replaces the old one
- transfers control to it

**Nomenclature**
- Because one program part is loaded over another in the same memory region.

---

## 🟢 Partition Allocation Schemes
In dynamic partitioned allocation, memory is treated as a set of variable-size free blocks and allocated blocks.

When a process or request needs memory, the OS must answer two questions:
- Which free block should be used?
- How should free space be tracked so allocation/deallocation stays fast?

### Objective
- fast allocation
- fast deallocation
### Tradeoff
- making allocation fast can make deallocation slower
- reducing wasted space can make searching slower
- improving fit quality can increase bookkeeping cost
### Balance
- speed
- fragmentation
- simplicity

---

## 🟢 Basic dynamic-partition model
Assume memory contains:
- OS region
- allocated partitions (“in use”)
- free partitions (“holes”)

The system keeps a free list of unused partitions.

Example:
- free block at a = 16M
- free block at c = 14M
- free block at e = 5M
- free block at g = 30M

These are free blocks of different sizes.

When a request arrives, the allocator chooses one of these blocks, gives part or all of it to the request, and if any space remains, returns the leftover to the free list.

When a block is freed, adjacent free blocks should be coalesced (merged), otherwise external fragmentation becomes severe.

---

## 🟢 First fit, best fit, worst fit

### A. First fit (FF)
Rule: Choose the first free partition that is large enough.

#### Allocation behavior
- scan the free list from the beginning
- stop at the first block that can satisfy the request
- allocate the needed amount
- return any leftover part to the free list

#### Deallocation behavior
- when a block is freed, combine it with adjacent free partitions if possible

#### Strength
- simple
- usually fast
- does not require scanning the entire list

#### Weakness
- can leave many small holes near the front of memory or front of the list
- over time may increase fragmentation
---
### B. Best fit (BF)
Rule: Choose the free partition that leaves the smallest leftover space.

So among all holes large enough, pick the tightest fit.

#### Allocation behavior
- search for the smallest usable block
- allocate from it
- leftover, if any, is returned to the free list

#### Deallocation behavior
- same as first fit: merge adjacent free neighbors when possible

#### Strength
- tries to reduce immediate wasted space
- often gives a more “efficient-looking” allocation per request

#### Weakness
- usually requires more searching
- tends to create many tiny leftover fragments that may be unusable later
---
### C. Worst fit
Rule: Choose the largest available free block.

#### Intuition
The hope is that carving from the largest block leaves a still-large remainder, which might be useful later.

#### Weakness
- often wastes large blocks
- usually not preferred in practice
- may leave medium-sized fragments everywhere

---

### D. Example
#### Free list:
- a = 16M
- c = 14M
- e = 5M
- g = 30M

#### Request:
- 13M

### First fit
First usable block is a = 16M
- allocate 13M there
- leftover = 3M

### Best fit
Smallest usable block is c = 14M
- allocate 13M there
- leftover = 1M

### Worst fit
Largest usable block is g = 30M
- allocate 13M there
- leftover = 17M

This shows the policy tradeoff:
- first fit is fast
- best fit minimizes immediate leftover
- worst fit preserves smaller holes but spends the largest block

---

### E. Deallocation & Merge Unused Blocks
Suppose one partition is freed and its neighboring regions are also free.

If the OS keeps them separate, memory becomes:
- many small pieces
- even when the total free memory is large
This is **external fragmentation.**

So deallocation usually includes:
- mark block free
- check left neighbor
- check right neighbor
- merge contiguous free blocks into one larger free partition

This process is called:
- **coalescing**
- **merging adjacent free partitions**

Without coalescing, dynamic partitioning degrades badly over time.

---

## 🟢 Choosing Free Blocks Fast

### Address-ordered vs size-ordered free lists

It helps to separate the two ideas clearly.

#### Free list ordered by address
##### Good for
- finding adjacent neighbors during deallocation
- easy coalescing

##### Bad for
- slower best-fit search
- may require scanning many entries

#### Free list ordered by size
##### Good for
- quickly finding a block of the right size
- supports best-fit more efficiently

##### Bad for
- harder to locate adjacent blocks in memory
- deallocation/coalescing becomes more expensive

Many real allocators therefore maintain more than one structure.

---

### Size-class lists

The last slide introduces a more advanced idea.

Instead of one sorted free list, keep multiple lists, one for each size class.

Example size classes:
- 1K
- 2K
- 4K
- 8K
- 16K
- 32K
- 64K
- 128K
- 256K
- 512K
- 1M

Each entry points to a linked list of free blocks of approximately that size.

The slide calls this “a kind of hash table,” but conceptually it is more like bucketed free lists or segregated free lists.

#### How it works
##### Allocation
If the request needs, say, 20K:
- look in the 32K bucket, or the smallest bucket large enough
- if available, remove one block quickly
- if not, move to a larger bucket and possibly split a larger block

##### Deallocation
When a block is freed:
- insert it back into the appropriate size bucket
- optionally coalesce it with its buddy/neighbor
- if merged, move it into a larger size class

##### Benefit
- allocation can be very fast
- avoids scanning one long free list
- supports many common request sizes efficiently

##### Cost
- more bookkeeping
- possible internal waste if the bucket size is larger than the request
- more complicated coalescing logic

---

### Variable-size vs fixed-size blocks
#### Variable-size blocks
##### Pros
- closer fit to request
- less internal fragmentation

##### Cons
- harder management
- external fragmentation
- more complicated coalescing

This is the classic dynamic partition approach.

#### Fixed-size blocks
##### Pros
- simple allocation/deallocation
- easy free-list management
- very fast

##### Cons
- more internal fragmentation
- may waste memory when requests vary widely

**Fixed-size blocks are common in:**
kernel object allocators
slab allocators
page allocators

---

## 🟢 Buddy System
The buddy system is a dynamic memory allocation technique that organizes free memory in blocks whose sizes are powers of 2.

Typical block sizes:
- 1K, 2K, 4K, 8K, 16K, ...
- up to some maximum block size, such as 1M

Its main goal is to make:
- **allocation fast**
- **deallocation and coalescing fast**

It is especially useful because it makes it easy to find the neighboring free block that could be merged.

---

### Core idea

Memory is managed as a hierarchy of blocks:
- a large block can be split into two equal halves
- those two halves are called buddies

#### Example:
- a 1M block can split into two 512K buddy blocks
- a 512K block can split into two 256K buddy blocks
- a 256K block can split into two 128K buddy blocks

When a block is freed:
- if its buddy is also free
- the two are merged back into one larger block

This merging can continue upward recursively.

That is why the buddy system is good at **fast coalescing.**

---

### Power of 2

The buddy system restricts block sizes to powers of 2 because that gives a very regular structure.

If a request is for some size S, the system allocates the smallest power-of-2 block that can hold it.

#### Example:
request 70K → allocate 128K
request 200K → allocate 256K
request 500K → allocate 512K

Consider a new 70k process is created
128k will be requested (why?)

Because 128K is the smallest power of 2 that is at least 70K.

In [2]:
import math

def solve_exponent(base, result):
    if base <= 0 or base == 1:
        raise ValueError("Base must be greater than 0 and not equal to 1.")
    if result <= 0:
        raise ValueError("Result must be greater than 0.")

    x = math.log(result, base)
    return x

# Example
base = 2
result = 70000

x = solve_exponent(base, result)
print(f"x = {x}")

x = 16.095067301607052


In [9]:
128/2

64.0

---

### Allocation process
Scenario: Suppose a process needs 70K, and only have 1M available

#### Step 1: round up the request
The allocator rounds this up to:
- 128K
because 128/2 = 64, so 128K is the smallest allowed block size that fits 70K.
---
#### Step 2: look in the matching free list
**The allocator checks:**
- is there a free 128K block?

If yes:
- allocate it directly

If no:
- look for a larger free block
---
#### Step 3: split larger blocks until the right size is reached

##### Example:
- no 128K block is free
- no 256K block is free
- no 512K block is free
- **but a 1M block is free**

**So the allocator splits:**
- First split
    - 1M → two 512K buddies
    - say at addresses 0 and 512K

- Second split
    - Take one 512K block and split it
    - 512K → two 256K buddies
    - say at addresses 0 and 256K

- Third split
    - Take one 256K block and split it
    - 256K → two 128K buddies
    - say at addresses 0 and 128K

Now allocate one of those 128K blocks to the process.

---

### Deallocation process
This is where the buddy system is especially strong. When a block is freed:
- put it back into its size class
- check whether its buddy is also free
- if yes, merge the two buddies into one larger block
- then check whether the larger merged block’s buddy is free
- repeat upward as long as possible

This recursive merge is called coalescing.

---

### Why is neighbor search fast?

In a general free-list allocator, when a block is freed, the system may need to search for adjacent free blocks to merge with.

**That can be slow.**

In the buddy system:
- a block’s only merge candidate is its buddy
- and the buddy must be the same size
- so you only check the free list for that size

So instead of searching all free memory, you search a very specific place.

---

### Disadvantages of the buddy system
1. Internal fragmentation

Because requests are rounded up to powers of 2, memory may be wasted inside allocated blocks.

**Example:**
- request 70K
- get 128K
- waste = 58K inside the block

This is the biggest downside.

2. Restrictive block sizes
- A program cannot get exactly the amount it asks for unless that amount is already a power of 2.

3. Can still fragment in practice
- Even though coalescing is easy, free memory may still be spread across sizes in a way that prevents large allocations until the right buddies become free.

### Internal vs external fragmentation in buddy system
Internal fragmentation - Very important in buddy allocation.

Example:
- request 33K → allocate 64K
- request 70K → allocate 128K

Unused memory inside the allocated block is internal fragmentation.

### External fragmentation
- Reduced compared with arbitrary variable partitions, because buddies can merge neatly.
But large-block availability can still be limited if free memory is split across many allocated buddies.

---

### Enhamcement Techniques
#### Compaction
Compaction means moving processes in memory so that scattered free holes are combined into one larger contiguous free block.

This is done to reduce external fragmentation.

**Why it is needed**

With dynamic partition allocation, memory can end up like this:
- process A
- free hole
- process B
- free hole
- process C
- free hole

Even if the total free memory is large, there may be no single contiguous block big enough for a new process.

Compaction fixes this by shifting allocated processes together, producing something like:
- process A
- process B
- process C
- one large free block

**Disadvantage**
Moving a process space in memory is slow/expensive, because:
- copying large amounts of memory
- updating process location information
- pausing or coordinating running processes
- possibly changing address mappings or references

So compaction is a way to reclaim usable contiguous memory.
---
### Swapping
Swapping means moving a process, or part of it, between:
- main memory
- secondary storage (disk/SSD)

Copy all (or some) of process out of main memory to secondary storage. This allows the OS to free RAM when memory is tight.


That means the OS keeps track of:
- which process is not currently in RAM
- where its image is stored on disk
- what memory state must be restored later

So the process is not destroyed; it is temporarily removed from main memory.

When it is needed again, the OS can:
- bring it back into memory
- restore its state
- resume execution

---